In [ ]:
from unienv_droid_data.raw import RawDroidDataset
import numpy as np
from pyzed import sl
import mediapy
import matplotlib.pyplot as plt
import cv2

from xbarray.transformations.pointcloud.numpy import (
    world_to_pixel_coordinate_and_depth
)
from xbarray.transformations.rotation_conversions.numpy import (
    euler_angles_to_matrix,
    matrix_to_quaternion
)

import transforms3d as tr3d

In [ ]:
path_to_target_dir = "/home/fang-lab-share/droid_raw_v1.0.1/" # Replace with the path where you downloaded the droid raw data
path_to_annotation_dir = "/home/fang-lab-share/droid_annotations" # Replace with the path where you placed the droid annotations

dataset = RawDroidDataset(
    root_dir=path_to_target_dir, 
    droid_annotations_path=path_to_annotation_dir,
    target_resolution=sl.Resolution(672, 376), 
    success_only=True, # Only load successful episodes
    read_right=False # Whether to read additional right camera images from the stereo camera
)

# Access dataset properties
print("Raw dataset length:", len(dataset))
print("Raw dataset space:", dataset.single_space)
print("Metadata space:", dataset.single_metadata_space)

In [ ]:
def droid_extrinsic_to_opencv(extrinsic : np.ndarray) -> np.ndarray:
    """Convert droid extrinsic matrix to OpenCV coordinate system.
    
    Args:
        extrinsic (np.ndarray): (.., 6) extrinsic containing cam2base position and euler angles from droid dataset.
        
    Returns:
        np.ndarray: Converted 4x4 extrinsic matrix in OpenCV coordinate system.
    """
    positions = extrinsic[..., :3]
    rotations = euler_angles_to_matrix(
        extrinsic[..., 3:][..., ::-1],  # Convert from XYZ to ZYX
        convention="ZYX"
    ) # PyTorch3D, when given convention="ZYX", applies rotations in X, then Y, then Z order
    extrinsic_opencv = np.broadcast_to(np.eye(4).reshape([1] * len(extrinsic.shape[:-1]) + [4, 4]), 
        list(extrinsic.shape[:-1]) + [4, 4]
    ).copy()
    extrinsic_opencv[..., :3, :3] = rotations
    extrinsic_opencv[..., :3, -1] = positions
    extrinsic_opencv = np.linalg.inv(extrinsic_opencv)
    return extrinsic_opencv

def overlay_eef_on_video(
    video : np.ndarray,
    cartesian_positions : np.ndarray,
    intrinsics : np.ndarray,
    extrinsics : np.ndarray,
    axis_length : float = 0.05,
) -> np.ndarray:
    """Overlay end-effector positions on video.
    
    Args:
        video (np.ndarray): (T, H, W, 3) video frames.
        cartesian_positions (np.ndarray): (T, 6) end-effector positions and euler angles in world coordinates.
        intrinsics (np.ndarray): (3, 3) camera intrinsic matrix.
        extrinsics (np.ndarray): (3, 4) camera extrinsic matrices.
        
    Returns:
        np.ndarray: Video frames with overlaid end-effector positions.
    """
    video_with_eef = video.copy()

    eef_rotations = euler_angles_to_matrix(
        cartesian_positions[..., 3:][..., ::-1],  # Convert from XYZ to ZYX
        convention="ZYX"
    )
    eef_origins = cartesian_positions[..., :3]
    eef_x_axes = eef_origins + axis_length * eef_rotations[..., 0]
    eef_y_axes = eef_origins + axis_length * eef_rotations[..., 1]
    eef_z_axes = eef_origins + axis_length * eef_rotations[..., 2]

    eef_points_world = np.concatenate([
        eef_origins,
        eef_x_axes,
        eef_y_axes,
        eef_z_axes,
    ], axis=0)  # (4 * T, 3)
    eef_2d_world, eef_depths = world_to_pixel_coordinate_and_depth(
        eef_points_world,
        intrinsics,
        extrinsics,
    )
    eef_2d_world = eef_2d_world.reshape(4, video.shape[0], 2)  # (4, T, 2)
    eef_origins_2d = eef_2d_world[0]  # (T, 2)
    eef_x_axes_2d = eef_2d_world[1]
    eef_y_axes_2d = eef_2d_world[2]
    eef_z_axes_2d = eef_2d_world[3]

    for t in range(video.shape[0]):
        origin_pixel = tuple(eef_origins_2d[t].astype(int))
        x_axis_pixel = tuple(eef_x_axes_2d[t].astype(int))
        y_axis_pixel = tuple(eef_y_axes_2d[t].astype(int))
        z_axis_pixel = tuple(eef_z_axes_2d[t].astype(int))

        cv2.line(video_with_eef[t], origin_pixel, x_axis_pixel, (255, 0, 0), 2) # X axis in red
        cv2.line(video_with_eef[t], origin_pixel, y_axis_pixel, (0, 255, 0), 2) # Y axis in green
        cv2.line(video_with_eef[t], origin_pixel, z_axis_pixel, (0, 0, 255), 2) # Z axis in blue
    return video_with_eef

In [ ]:
# Access data
first_step_data = dataset[0]

# Access slices of data
first_ten_steps = dataset[:10]

# Access data through indexes
indexed_data = dataset[np.array([0, 5, 10, 15])]

# Access data and metadata
first_step_data, first_step_metadata = dataset.get_at_with_metadata(0)

# Access slice of data and metadata
first_ten_steps_data, first_ten_steps_metadata = dataset.get_at_with_metadata(slice(10))

In [ ]:
# Access trajectory (trajectory doesn't support slices)
print("Trajectory length:", dataset.num_trajectories)
first_trajectory_data = dataset.get_trajectory_at(2)
# first_trajectory_data, first_trajectory_metadata = dataset.get_trajectory_at_with_metadata(0)

In [ ]:
vis_camera = 1

extrinsics = first_trajectory_data['steps']['observation']['camera_extrinsics'][f'exterior_camera_{vis_camera}_left'][0]
extrinsics_opencv = droid_extrinsic_to_opencv(extrinsics)
print(extrinsics, extrinsics_opencv, first_trajectory_data['steps']['observation']['camera_extrinsics'][f'exterior_camera_{vis_camera}_is_postcalibrated'][0])
intrinsics = first_trajectory_data['steps']['observation']['camera_intrinsics'][f'exterior_camera_{vis_camera}_left'][0]
vids = first_trajectory_data['steps']['observation'][f'exterior_image_{vis_camera}_left']
vids_overlaid = overlay_eef_on_video(
    vids,
    first_trajectory_data['steps']['observation']['cartesian_position'],
    intrinsics,
    extrinsics_opencv,
    axis_length=0.05
)

mediapy.show_video(vids_overlaid, fps=15)

In [ ]:
first_trajectory_data['steps']['observation']['camera_extrinsics']['exterior_camera_1_is_postcalibrated']

In [ ]:
import viser
viser_instance = viser.ViserServer()

In [ ]:
viser_instance.scene.remove_by_name("exterior_camera_1_left")
viser_instance.scene.remove_by_name("exterior_camera_2_left")
extrinsics_1 = first_trajectory_data['steps']['observation']['camera_extrinsics'][f'exterior_camera_{1}_left'][0]
extrinsics_2 = first_trajectory_data['steps']['observation']['camera_extrinsics'][f'exterior_camera_{2}_left'][0]
extrinsics_vis_1 = viser_instance.scene.add_camera_frustum(
    "exterior_camera_1_left",
    fov=90,
    aspect=16/9,
    image=first_trajectory_data['steps']['observation'][f'exterior_image_1_left'][0],
    wxyz=matrix_to_quaternion(euler_angles_to_matrix(
        extrinsics_1[3:][..., ::-1],  # Convert from XYZ to ZYX
        convention="ZYX"
    )),
    position=extrinsics_1[:3],
)
extrinsics_vis_2 = viser_instance.scene.add_camera_frustum(
    "exterior_camera_2_left",
    fov=90,
    aspect=16/9,
    image=first_trajectory_data['steps']['observation'][f'exterior_image_2_left'][0],
    wxyz=matrix_to_quaternion(euler_angles_to_matrix(
        extrinsics_2[3:][..., ::-1],  # Convert from XYZ to ZYX
        convention="ZYX"
    )),
    position=extrinsics_2[:3],
)